<a href="https://colab.research.google.com/github/adriantang23/cs505-machine-translation/blob/main/opus_mt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Installs to get OPUS-MT
!pip install transformers sacrebleu sentencepiece

In [ ]:
# Mount drive to retrieve datasets
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# Parse SGM files, using re import, so it is interpretable for model
import re
import sacrebleu
from tqdm import tqdm # Using tqdm to track eval progress

def parse_sgm(filepath):
    sentences = []
    with open(filepath, encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line.startswith('<seg'):
                match = re.search(r'<seg[^>]*>(.*?)</seg>', line)
                if match:
                    sentences.append(match.group(1).strip())
    return sentences

fr_test_path = '/content/drive/MyDrive/cs505_data/newstest2014-fren-src.fr.sgm'
en_test_path = '/content/drive/MyDrive/cs505_data/newstest2014-fren-ref.en.sgm'

test_fr = parse_sgm(fr_test_path)
test_en = parse_sgm(en_test_path)

print(f"FR: {test_fr[0]}")
print(f"EN: {test_en[0]}")



In [ ]:
from transformers import MarianMTModel, MarianTokenizer

model_name = "Helsinki-NLP/opus-mt-fr-en"
model = MarianMTModel.from_pretrained(model_name)
tokenizer = MarianTokenizer.from_pretrained(model_name)
print("Model loaded successfully")

In [ ]:
# BLEU Eval
def translate_opus(sentences, batch_size=32):
    translations = []
    for i in tqdm(range(0, len(sentences), batch_size)):
        batch = sentences[i:i+batch_size]
        inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=512)
        translated = model.generate(**inputs)
        decoded = [tokenizer.decode(t, skip_special_tokens=True) for t in translated]
        translations.extend(decoded)
    return translations

translations_opus = translate_opus(test_fr)

bleu = sacrebleu.corpus_bleu(translations_opus, [test_en])
print(f"\nOPUS-MT Transformer BLEU score: {bleu.score:.2f}")

OPUS-MT Transformer BLEU score: 37.99

Leaving here if outputs don't save;
Training took more than an hour.

In [ ]:
# Tests

print("\nSample translations:")
for i in range(5):
    print(f"\nFR:  {test_fr[i]}")
    print(f"REF: {test_en[i]}")
    print(f"OPUS: {translations_opus[i]}")

